# Assignment 6: K-Means Clustering Implementation
**Student ID:** 220119

This notebook implements **K-Means clustering** with `scikit-learn`, training on the
**Teens Mental Health Dataset** (the same dataset used in Labs 2, 3, and 4-Assignment 5) and
then assigning **10 custom teen profiles** to the learned clusters as a real-world test.

## Datasets
* **Standard training set:** Teens Mental Health Dataset (1 200 rows, 13 columns).
  Features used (12 of them — the binary `depression_label` target is dropped because
  clustering is unsupervised): `age`, `gender`, `daily_social_media_hours`,
  `platform_usage`, `sleep_hours`, `screen_time_before_sleep`, `academic_performance`,
  `physical_activity`, `social_interaction_level`, `stress_level`, `anxiety_level`,
  `addiction_level`.
* **Custom real-world set:** 10 hand-crafted teen profiles with identical column names
  and data types, so the same fitted `StandardScaler` can be re-applied at prediction time.

## Pipeline
1. `!git clone` the GitHub repository (or fall back to the local copy) so the dataset
   and the custom CSV are available in Colab with no manual file upload.
2. Load the Teens Mental Health dataset, perform EDA, encode the three categorical
   columns with `LabelEncoder`.
3. Standardise every feature with `StandardScaler` (K-Means uses Euclidean distance).
4. Use the **Elbow method** to choose the optimal $K$ (range 1–10).
5. Fit `KMeans` with the chosen $K$, save the model and the scaler with `joblib`.
6. Load the 10 custom teen profiles, scale them with the **same** scaler (no
   re-fitting), and use `model.predict` to assign each to a cluster.
7. Produce the four required visuals: elbow curve, cluster scatter (PCA projection),
   prediction table, and a cluster interpretation text cell.

## 0. Imports and Configuration

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import joblib

SCREENSHOT_DIR = 'screenshots'
MODEL_DIR = 'model'
DATA_DIR = 'dataset'
for d in (SCREENSHOT_DIR, MODEL_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150})
sns.set_style('whitegrid')
print('Environment ready.')

## 1. Automatic Data Retrieval (no manual upload)

The notebook first tries the local `dataset/` folder, otherwise `!git clone`s the public
GitHub repository so that **Run All** works on Colab end-to-end with no manual file
uploads.

In [ ]:
GITHUB_REPO = 'https://github.com/ahsanjust/Artificial-Intelligence-and-Machine-Learning-Lab'
REPO_DIR = 'Artificial-Intelligence-and-Machine-Learning-Lab'

def ensure_dataset():
    # Already in place locally?
    if os.path.exists(os.path.join(DATA_DIR, 'Teen_Mental_Health_Dataset.csv')):
        return
    if not os.path.exists(REPO_DIR):
        print('Cloning repository...')
        os.system(f'git clone --depth 1 {GITHUB_REPO}.git')
    # Sync required files into the working tree so subsequent cells can find them.
    src_dir = os.path.join(REPO_DIR, 'Lab_4', 'KMeans', 'dataset')
    for fname in ('Teen_Mental_Health_Dataset.csv', 'Custom_Teen_Profiles.csv'):
        src = os.path.join(src_dir, fname)
        dst = os.path.join(DATA_DIR, fname)
        if os.path.exists(src) and not os.path.exists(dst):
            os.makedirs(DATA_DIR, exist_ok=True)
            import shutil
            shutil.copy(src, dst)

ensure_dataset()
print('Datasets in', DATA_DIR, ':', sorted(os.listdir(DATA_DIR)))

## 2. Load the Standard Training Data

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'Teen_Mental_Health_Dataset.csv'))
print('Shape:', df.shape)
display(df.head())

print('\nDataset info:')
df.info()

print('\nStatistical summary:')
display(df.describe(include='all').T)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Distributions of all numeric features

In [ ]:
num_cols = ['age', 'daily_social_media_hours', 'sleep_hours',
            'screen_time_before_sleep', 'academic_performance',
            'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level']
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=20, color=sns.color_palette('tab10')[i % 10],
                 edgecolor='black', alpha=0.85)
    axes[i].set_title(f'Distribution of {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Feature_Distributions.png'), bbox_inches='tight')
plt.show()

### 3.2 Categorical counts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cat_cols = ['gender', 'platform_usage', 'social_interaction_level']
for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, palette='pastel', ax=ax)
    ax.set_title(f'Count of {col}')
    for p in ax.patches:
        ax.text(p.get_x() + p.get_width() / 2, p.get_height() + 2,
                f'{int(p.get_height())}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Categorical_Counts.png'), bbox_inches='tight')
plt.show()

### 3.3 Correlation Heatmap

In [ ]:
df_corr = df.copy()
for col in cat_cols:
    df_corr[col] = LabelEncoder().fit_transform(df_corr[col])
plt.figure(figsize=(10, 8))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap (with encoded categoricals)')
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Correlation_Heatmap.png'), bbox_inches='tight')
plt.show()

## 4. Preprocessing — Feature Encoding and Scaling

K-Means is an **unsupervised** algorithm, so the binary target `depression_label` is
**dropped** before clustering. The three categorical columns are encoded with
`LabelEncoder`, and then every feature is standardised with `StandardScaler` so that
all features contribute equally to the Euclidean distance.

In [ ]:
# Drop the target (clustering is unsupervised).
features = [
    'age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours',
    'screen_time_before_sleep', 'academic_performance', 'physical_activity',
    'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level'
]
X = df[features].copy()

label_encoders = {}
for col in ['gender', 'platform_usage', 'social_interaction_level']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

print('Encoding maps:')
for col, le in label_encoders.items():
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.values)
print('\nScaled matrix shape :', X_scaled.shape)
print('Scaled matrix mean   :', X_scaled.mean(axis=0).round(4))
print('Scaled matrix std    :', X_scaled.std(axis=0).round(4))

## 5. Optimal K Selection — The Elbow Method

We fit K-Means for $K = 1, \dots, 10$ and record the **Within-Cluster Sum of Squares
(WCSS / Inertia)** and the **Silhouette score**. The elbow is the value of $K$ where
WCSS stops decreasing sharply. The silhouette score peaks at the most cohesive
partition.

In [ ]:
wcss, sil_scores, K_range = [], [], range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_) if k > 1 else np.nan
    sil_scores.append(sil)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(K_range, wcss, 'o-', linewidth=2, color='#4C72B0')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('WCSS (Inertia)')
axes[0].set_title('Elbow Method')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, sil_scores, 's-', linewidth=2, color='#C44E52')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Elbow_Curve.png'), bbox_inches='tight')
plt.show()

print('K  |  WCSS       |  Silhouette')
print('---+-------------+-------------')
for k, w, s in zip(K_range, wcss, sil_scores):
    print(f'{k:>2} | {w:>10.2f}  |  {s if np.isnan(s) else round(s, 4)}')

The elbow curve flattens visibly between $K=3$ and $K=5$. The silhouette score peaks
at $K = 4$ (silhouette $\approx 0.0695$). The maximum absolute silhouette on this dataset
is low ($\sim 0.07$) — this is typical of real-world survey data with overlapping
feature distributions. We select **K = 4** because it is the global maximum of the
silhouette score, which is the most principled automated criterion.

In [ ]:
OPTIMAL_K = int(np.nanargmax(sil_scores)) + 1   # argmax is on K = 1..10, shift by 1
print(f'Optimal K chosen by silhouette maximum: {OPTIMAL_K}')

## 6. K-Means — Model Fitting and Persistence

In [ ]:
kmeans = KMeans(n_clusters=OPTIMAL_K, n_init=10, random_state=SEED)
cluster_labels = kmeans.fit_predict(X_scaled)
df['Cluster'] = cluster_labels

# Save the fitted model and the scaler so the same objects can be re-used for the custom data.
joblib.dump(kmeans, os.path.join(MODEL_DIR, '220119_kmeans.pkl'))
joblib.dump(scaler, os.path.join(MODEL_DIR, '220119_scaler.pkl'))
print('Model + scaler saved to', MODEL_DIR)
print('\nCluster counts:')
print(df['Cluster'].value_counts().sort_index())

## 7. Visualisation — Cluster Scatter Plot (PCA Projection)

Because the feature space is 12-dimensional, we project the scaled data onto its first
two **principal components** for a 2D scatter plot. The centroids are shown as black
crosses so the plot is interpretable.

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
centroids_pca = pca.transform(kmeans.cluster_centers_)

fig, ax = plt.subplots(figsize=(10, 7))
palette = sns.color_palette('tab10', n_colors=OPTIMAL_K)
for c in range(OPTIMAL_K):
    mask = cluster_labels == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               s=30, color=palette[c], edgecolor='k', alpha=0.7,
               label=f'Cluster {c}')
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           s=320, marker='X', c='black', edgecolor='yellow', linewidth=2,
           label='Centroid')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('K-Means Clusters on PCA Projection of the Scaled Features')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Cluster_Scatter.png'), bbox_inches='tight')
plt.show()

## 8. Cluster Interpretation (centroid profile in original units)

In [ ]:
# Recover the centroid in the original (unscaled) feature space and use the *mode*
# of the categoricals for a faithful centroid profile.
centroids_scaled = kmeans.cluster_centers_
centroids_num = pd.DataFrame(
    scaler.inverse_transform(centroids_scaled),
    columns=features
)

rows = []
for c in range(OPTIMAL_K):
    row = {
        'Age': centroids_num.iloc[c]['age'].round(1),
        'Gender': df.loc[cluster_labels == c, 'gender'].mode().iloc[0],
        'Daily Social Media (h)': centroids_num.iloc[c]['daily_social_media_hours'].round(1),
        'Platform': df.loc[cluster_labels == c, 'platform_usage'].mode().iloc[0],
        'Sleep (h)': centroids_num.iloc[c]['sleep_hours'].round(1),
        'Screen before sleep (h)': centroids_num.iloc[c]['screen_time_before_sleep'].round(1),
        'Academic perf.': centroids_num.iloc[c]['academic_performance'].round(2),
        'Physical activity': centroids_num.iloc[c]['physical_activity'].round(1),
        'Social interaction': df.loc[cluster_labels == c, 'social_interaction_level'].mode().iloc[0],
        'Stress (1-10)': centroids_num.iloc[c]['stress_level'].round(1),
        'Anxiety (1-10)': centroids_num.iloc[c]['anxiety_level'].round(1),
        'Addiction (1-10)': centroids_num.iloc[c]['addiction_level'].round(1),
        'Size': (cluster_labels == c).sum(),
    }
    rows.append(row)

centroids = pd.DataFrame(rows, index=range(OPTIMAL_K))
centroids.index.name = 'Cluster'
display(centroids)

**Cluster interpretation** (centroid values printed above):

* **Cluster 0 — "Older Male Teens with Elevated Addiction":** oldest cluster
  (mean age 17.2), predominantly male, the highest `addiction_level` of all clusters
  (6.3) and the highest `screen_time_before_sleep` (2.0 h). The most social-media-engaged
  group.
* **Cluster 1 — "High-Stress Female Teens":** all-female cluster with the
  **highest `stress_level` (8.2)** of all clusters and the second-highest
  `addiction_level` (5.7). These teens are juggling many pressures.
* **Cluster 2 — "Younger Male Teens with Low Anxiety":** youngest cluster (mean
  age 14.6), male-dominant, with the **lowest `anxiety_level` (4.8)** and
  `addiction_level` (4.9). The least digitally-addicted group.
* **Cluster 3 — "Balanced Female Teens with Low Stress":** predominantly female,
  mean age 16, with the **lowest `stress_level` (3.2)** of all clusters. The most
  relaxed, balanced profile.

> **Caveat.** The maximum silhouette score on this dataset is only $\sim 0.07$, so the
> clusters overlap substantially in the original 12-D space. The above profiles are best
> read as *rough tendencies* of the dominant demographic in each cluster, not as crisp
> hard rules.

## 9. Real-World Prediction — Custom Survey Data

We load the 10 hand-crafted teen profiles, scale them with the **same fitted
`StandardScaler`** used on the training set (no re-fitting) and assign each to a
cluster with `model.predict`. The custom profiles have identical column names and
dtypes to the training set, so the same scaler applies directly.

In [ ]:
custom = pd.read_csv(os.path.join(DATA_DIR, 'Custom_Teen_Profiles.csv'))
print('Custom data shape:', custom.shape)
display(custom)

# Encode the categoricals with the SAME LabelEncoders used at training time.
custom_enc = custom.copy()
for col in ['gender', 'platform_usage', 'social_interaction_level']:
    custom_enc[col] = label_encoders[col].transform(custom_enc[col])

X_custom_scaled = scaler.transform(custom_enc[features].values)  # use SAME scaler, do NOT re-fit
custom_clusters = kmeans.predict(X_custom_scaled)
custom['Assigned_Cluster'] = custom_clusters
display(custom)

# Plot custom profiles on the same PCA axes for a clean overlay.
custom_pca = pca.transform(X_custom_scaled)
plt.figure(figsize=(10, 7))
for c in range(OPTIMAL_K):
    mask = cluster_labels == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                s=25, color=palette[c], edgecolor='none', alpha=0.25)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
            s=260, marker='X', c='black', edgecolor='yellow', linewidth=2,
            label='Centroid')
plt.scatter(custom_pca[:, 0], custom_pca[:, 1],
            s=180, c='red', edgecolor='black', marker='*', label='Custom profile')
for i, c in enumerate(custom_clusters):
    plt.annotate(f'#{i+1} → {c}',
                 (custom_pca[i, 0], custom_pca[i, 1]),
                 textcoords='offset points', xytext=(8, 6), fontsize=9)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Real-World Prediction — Custom Teen Profiles on the Cluster Map')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Custom_Prediction_Plot.png'), bbox_inches='tight')
plt.show()

### 9.1 Predicted Cluster Distribution for Custom Data

In [ ]:
print(custom['Assigned_Cluster'].value_counts().sort_index())

plt.figure(figsize=(7, 4))
sns.countplot(x=custom['Assigned_Cluster'], palette='tab10')
plt.xlabel('Assigned Cluster')
plt.ylabel('Number of Custom Profiles')
plt.title('Distribution of Custom Profiles Across Clusters')
for i, v in enumerate(custom['Assigned_Cluster'].value_counts().sort_index().values):
    plt.text(i, v + 0.05, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Custom_Cluster_Distribution.png'), bbox_inches='tight')
plt.show()

## 10. Per-cluster summary in original units

In [ ]:
# Use the *encoded* version of the data so the groupby mean works on every column.
df_enc = X.copy()
df_enc['Cluster'] = cluster_labels
summary = df_enc.groupby('Cluster').mean().round(2)
summary['Count'] = df_enc.groupby('Cluster').size()
# Rename the encoded categorical columns for clarity.
summary = summary.rename(columns={
    'gender': 'gender (encoded)',
    'platform_usage': 'platform_usage (encoded)',
    'social_interaction_level': 'social_interaction_level (encoded)',
})
display(summary)

## 11. Reload the Saved Model and Verify the Pipeline

This cell demonstrates that the persisted `KMeans` and `StandardScaler` objects
reproduce the exact same cluster assignments for the custom data, without any
re-training.

In [ ]:
loaded_kmeans = joblib.load(os.path.join(MODEL_DIR, '220119_kmeans.pkl'))
loaded_scaler = joblib.load(os.path.join(MODEL_DIR, '220119_scaler.pkl'))

re_pred = loaded_kmeans.predict(loaded_scaler.transform(custom_enc[features].values))
assert (re_pred == custom_clusters).all(), 'Reload mismatch!'
print('Reload OK — same predictions as the in-memory model.')

## 12. Summary

* **Training set:** 1 200 teens from the Teens Mental Health dataset, 12 features
  (after dropping the binary target).
* **Optimal K:** 4, selected by the **global maximum of the silhouette score**
  ($K = 4 \Rightarrow \text{silhouette} \approx 0.0695$).
* **Real-world prediction:** 10 hand-crafted teen profiles were scaled with the same
  `StandardScaler` used at training time and assigned to clusters with `model.predict`.
* **Persistence:** both the scaler and the fitted K-Means are saved with `joblib` so the
  pipeline is fully reproducible.

### Required artefacts produced
* `screenshots/Feature_Distributions.png` — numeric feature histograms.
* `screenshots/Categorical_Counts.png` — categorical counts.
* `screenshots/Correlation_Heatmap.png` — feature correlation.
* `screenshots/Elbow_Curve.png` — WCSS + silhouette analysis.
* `screenshots/Cluster_Scatter.png` — clusters with centroids (PCA projection).
* `screenshots/Custom_Prediction_Plot.png` — custom profiles overlaid on the cluster map.
* `screenshots/Custom_Cluster_Distribution.png` — distribution of custom assignments.
* `model/220119_kmeans.pkl` and `model/220119_scaler.pkl` — saved pipeline.

### How to reproduce
1. Open the notebook in Google Colab.
2. Click **Runtime → Run all**.
3. The notebook clones the GitHub repo (or uses the local copy), fits the model, and
   predicts the cluster for each custom profile — no manual file uploads required.